# 🎙️ Whisper Transcriber (Google Colab)

Transcribe **audio and video files stored in Google Drive** into text using OpenAI's Whisper.

**Expected folder layout in Google Drive**
```
MyDrive/Notebooks/transcribeWhisper/
├── videos/       ← video files
├── sounds/       ← audio files
└── transcribes/  ← transcrip results
```

**Workflow**
1. Install dependencies and mount Google Drive
2. Choose your settings (model, language, task, output formats)
3. *(Optional)* Extract audio from videos
4. Load the Whisper model
5. Transcribe every audio file
6. Review the results

> 💡 Enable a GPU: **Runtime → Change runtime type → T4 GPU**.

## 1. Setup

Install Whisper and `ffmpeg`, then mount Google Drive.

In [ ]:
# Install Whisper (from PyPI) and ffmpeg
!pip install -q -U openai-whisper
!apt-get -qq update && apt-get -qq install -y ffmpeg

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

## 2. Settings


| Setting | Meaning |
|---|---|
| **MODEL** | Bigger = more accurate, slower. `.en` models are English-only. `turbo` is fast, near-large-quality multilingual model (transcription only). |
| **LANGUAGE** | Spoken language in the audio, or `Auto-detect`. Setting it explicitly is faster and more reliable. |
| **TASK** | `transcribe` keeps the original language; `translate` outputs English text. |
| **INITIAL_PROMPT** | Optional hint (names, technical terms) to improve spelling of specialised vocabulary. |
| **SAVE_*** | Which output formats to create. `srt`/`vtt` contain timestamps (subtitles). |
| **SKIP_EXISTING** | Don't redo files that already have all selected outputs. |

Model guide:
- `tiny` / `base` → quick drafts
- `small` / `medium` → good balance
- `large-v3` → best accuracy.

In [ ]:
#@title Settings { run: "auto" }

BASE_DIR = "/content/drive/MyDrive/Notebooks/transcribeWhisper"  #@param {type:"string"}

MODEL = "medium.en"  #@param ["tiny", "tiny.en", "base", "base.en", "small", "small.en", "medium", "medium.en", "large-v3", "turbo"]
LANGUAGE = "English"  #@param ["Auto-detect", "English", "Arabic", "Persian"]
TASK = "transcribe"  #@param ["transcribe", "translate"]
INITIAL_PROMPT = ""  #@param {type:"string"}

SAVE_TXT = True   #@param {type:"boolean"}
SAVE_SRT = False  #@param {type:"boolean"}
SAVE_VTT = False  #@param {type:"boolean"}
SAVE_JSON = False  #@param {type:"boolean"}

SKIP_EXISTING = True  #@param {type:"boolean"}

### Validate the settings


In [ ]:
import os
from pathlib import Path
import torch

# --- Language name -> Whisper language code (None = auto-detect) ---
LANGUAGE_CODES = {
    "Auto-detect": None, "English": "en", "Arabic": "ar", "Persian": "fa",
}
language_code = LANGUAGE_CODES[LANGUAGE]
model_name = MODEL

# --- English-only (.en) models cannot handle other languages or translation ---
if model_name.endswith(".en"):
    if language_code not in ("en",) or TASK == "translate":
        fixed = model_name[:-3]
        print(f"⚠️  '{model_name}' is English-only. Switching to the multilingual '{fixed}' model.")
        model_name = fixed

# --- The turbo model was not trained for translation ---
if model_name == "turbo" and TASK == "translate":
    print("⚠️  'turbo' does not support translation. Switching to 'large-v3'.")
    model_name = "large-v3"

# --- Folders ---
base_dir = Path(BASE_DIR)
videos_dir = base_dir / "videos"
sounds_dir = base_dir / "sounds"
transcribes_dir = base_dir / "transcribes"
for folder in (videos_dir, sounds_dir, transcribes_dir):
    folder.mkdir(parents=True, exist_ok=True)

# --- Output formats ---
output_formats = [fmt for fmt, on in
                  (("txt", SAVE_TXT), ("srt", SAVE_SRT), ("vtt", SAVE_VTT), ("json", SAVE_JSON)) if on]
if not output_formats:
    raise ValueError("Select at least one output format (SAVE_TXT, SAVE_SRT, ...).")

# --- Device ---
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    print("⚠️  No GPU detected - transcription will be slow. Use Runtime → Change runtime type → T4 GPU.")

print(f"Model     : {model_name}")
print(f"Language  : {LANGUAGE} ({language_code})")
print(f"Task      : {TASK}")
print(f"Formats   : {', '.join(output_formats)}")
print(f"Device    : {device}")
print(f"Base dir  : {base_dir}")

## 3. Extract audio from videos *(optional)*

Skip this section if you only have audio files.

Every video in `videos/` is converted to a **mono, 16 kHz MP3** in `sounds/` .
Videos that were already converted are skipped.

In [ ]:
import subprocess

VIDEO_EXTENSIONS = {".mp4", ".mkv", ".mov", ".avi", ".webm", ".m4v", ".flv", ".wmv"}

video_files = sorted(p for p in videos_dir.iterdir() if p.suffix.lower() in VIDEO_EXTENSIONS)
print(f"Found {len(video_files)} video file(s) in {videos_dir}\n")

for video_path in video_files:
    audio_path = sounds_dir / (video_path.stem + ".mp3")

    if audio_path.exists() and SKIP_EXISTING:
        print(f"⏭️  Already converted: {video_path.name}")
        continue

    print(f"🎬 Extracting audio: {video_path.name}")
    command = [
        "ffmpeg", "-y", "-loglevel", "error",
        "-i", str(video_path),
        "-vn",                 # drop the video stream
        "-ac", "1",            # mono
        "-ar", "16000",        # 16 kHz sample rate
        "-b:a", "64k",         # bitrate
        str(audio_path),
    ]
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"   ❌ ffmpeg failed: {result.stderr.strip()}")
    else:
        print(f"   ✅ Saved: {audio_path.name}")

## 4. Load the Whisper model


In [ ]:
import whisper

print(f"Loading '{model_name}' on {device} ...")
model = whisper.load_model(model_name, device=device)
print("✅ Model loaded.")

## 5. Transcribe

For every audio file in `sounds/` the notebook:
1. Transcribes it with the selected model, language and task
2. Saves the selected formats into `transcribes/` (UTF-8)
3. Records the time taken, and keeps going even if one file fails

Supported audio types: mp3, wav, m4a, flac, ogg, aac, wma, opus.

In [ ]:
import time
from tqdm.auto import tqdm
from whisper.utils import get_writer

AUDIO_EXTENSIONS = {".mp3", ".wav", ".m4a", ".flac", ".ogg", ".aac", ".wma", ".opus"}

audio_files = sorted(p for p in sounds_dir.iterdir() if p.suffix.lower() in AUDIO_EXTENSIONS)
print(f"Found {len(audio_files)} audio file(s) in {sounds_dir}\n")

def outputs_exist(audio_path):
    # True when every selected output format already exists for this audio file.
    return all((transcribes_dir / f"{audio_path.stem}.{fmt}").exists() for fmt in output_formats)

def transcribe_file(audio_path):
    # Run Whisper on a single file and return the result dictionary.
    options = dict(
        task=TASK,
        fp16=(device == "cuda"),   # half precision is faster on GPU, unsupported on CPU
        verbose=False,
    )
    if language_code:
        options["language"] = language_code
    if INITIAL_PROMPT.strip():
        options["initial_prompt"] = INITIAL_PROMPT.strip()
    return model.transcribe(str(audio_path), **options)

def save_result(result, audio_path):
    # Write the result in each selected format (txt / srt / vtt / json).
    for fmt in output_formats:
        writer = get_writer(fmt, str(transcribes_dir))
        writer(result, str(audio_path))

summary = []

for audio_path in tqdm(audio_files, desc="Files"):
    if SKIP_EXISTING and outputs_exist(audio_path):
        summary.append({"file": audio_path.name, "status": "skipped (exists)", "language": "-", "seconds": 0})
        continue

    start = time.time()
    try:
        result = transcribe_file(audio_path)
        save_result(result, audio_path)
        summary.append({
            "file": audio_path.name,
            "status": "done",
            "language": result.get("language", "-"),
            "seconds": round(time.time() - start, 1),
        })
    except Exception as error:
        summary.append({"file": audio_path.name, "status": f"failed: {error}", "language": "-", "seconds": 0})
        print(f"❌ {audio_path.name}: {error}")

print("\nFinished.")

## 6. Review the results

A summary table of what happened to each file, followed by a short preview of every `.txt` transcript.

In [ ]:
import pandas as pd
from IPython.display import display

display(pd.DataFrame(summary))

In [ ]:
# Preview the first 500 characters of each transcript.
PREVIEW_CHARS = 500

for txt_file in sorted(transcribes_dir.glob("*.txt")):
    text = txt_file.read_text(encoding="utf-8")
    print(f"📄 {txt_file.name}  ({len(text):,} characters)")
    print(text[:PREVIEW_CHARS] + (" ..." if len(text) > PREVIEW_CHARS else ""))
    print("-" * 60)

## 7. Free GPU memory *(optional)*

Run this when you are finished, or before switching to a different model.

In [ ]:
import gc

del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("🧹 Model removed from memory.")